### Setup

In [ ]:
!pip install yfinance mplfinance -q tensorflow==2.21.0

import yfinance as yf
import pandas as pd
import os
import time
import numpy as np
import random
import matplotlib.pyplot as plt
import mplfinance as mpf
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2, ResNet50, InceptionV3, MobileNetV3Small
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.callbacks import EarlyStopping
import warnings

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/TradeVisionAI'
DATASET_DIR = f'{BASE_DIR}/dataset_borse'
MODELS_DIR = f'{BASE_DIR}/modelli'
PLOTS_DIR = f'{BASE_DIR}/grafici'

for folder in [f'{DATASET_DIR}/COMPRA', f'{DATASET_DIR}/VENDI', MODELS_DIR, PLOTS_DIR]:
    os.makedirs(folder, exist_ok=True)

print(f"Setup completato. Cartella di base: {BASE_DIR}")

### Generazione dataset

In [ ]:
print("Inizio scaricamento dati reali e disegno dei grafici...")

tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'META', 'NVDA', 'NFLX']
window_size = 20
future_window = 5

count_compra = 0
count_vendi = 0

mc = mpf.make_marketcolors(up='green', down='red', edge='inherit', wick='black')
s = mpf.make_mpf_style(marketcolors=mc, gridstyle='')

for ticker in tickers:
    stock = yf.Ticker(ticker)
    data = stock.history(period="2y")

    if data.empty:
        continue

    for i in range(0, len(data) - window_size - future_window, 2):
        window_data = data.iloc[i : i+window_size]

        current_price = data['Close'].iloc[i+window_size - 1]
        future_price = data['Close'].iloc[i+window_size + future_window - 1]

        performance = (future_price - current_price) / current_price

        if performance > 0.03:
            label = "COMPRA"
            count_compra += 1
        elif performance < -0.03:
            label = "VENDI"
            count_vendi += 1
        else:
            continue

        filepath = f"{DATASET_DIR}/{label}/{ticker}_{i}.png"

        if not os.path.exists(filepath):
            mpf.plot(window_data,
                     type='candle',
                     style=s,
                     axisoff=True,
                     savefig=dict(fname=filepath, bbox_inches='tight', pad_inches=0))

print(f"Dataset pronto su Drive! Immagini COMPRA: {count_compra} | Immagini VENDI: {count_vendi}")

### Data Augmentation

In [ ]:
print("Preparazione dei tensori e Data Augmentation in corso...")

data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomRotation(0.05),
  tf.keras.layers.RandomZoom(0.1),
  tf.keras.layers.RandomContrast(0.2)
])

train_dataset = tf.keras.utils.image_dataset_from_directory(
  DATASET_DIR, validation_split=0.2, subset="training", seed=123,
  image_size=(224, 224), batch_size=16)

val_dataset = tf.keras.utils.image_dataset_from_directory(
  DATASET_DIR, validation_split=0.2, subset="validation", seed=123,
  image_size=(224, 224), batch_size=16)

for images, labels in val_dataset.take(1):
    sample_img = np.expand_dims(images[0], axis=0)
    break

print("Dataset caricato in memoria per l'addestramento.")

### MobileNet V2 + quantizzazione

In [ ]:
print("Costruzione architettura Edge AI (MobileNetV2)...")

base_model_v2 = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model_v2.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model_v2(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
outputs = Dense(2, activation='softmax')(x)

model_v2 = Model(inputs, outputs)
model_v2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Inizio addestramento MobileNetV2 (5 epoche)...")
history_v2 = model_v2.fit(train_dataset, validation_data=val_dataset, epochs=5)

v2_keras_path = f'{MODELS_DIR}/mobilenet_model'
model_v2.export(v2_keras_path)

print("\nApplicazione Post-Training Quantization (Float32 -> Int8)...")
converter = tf.lite.TFLiteConverter.from_keras_model(model_v2)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()

tflite_path = f'{MODELS_DIR}/mobilenet_v2_quant.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_quant_model)

print(f"Modello Edge salvato su Drive: {tflite_path}")

### Inception V3

In [ ]:
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess

print("Costruzione Modello InceptionV3...")
base_model_inception = InceptionV3(input_shape=(224, 224, 3), include_top=False, weights='imagenet')

base_model_inception.trainable = True
for layer in base_model_inception.layers[:-50]:
    layer.trainable = False

inputs_inc = tf.keras.Input(shape=(224, 224, 3))
x_inc = data_augmentation(inputs_inc)
x_inc = inception_preprocess(x_inc)
x_inc = base_model_inception(x_inc, training=True)
x_inc = GlobalAveragePooling2D()(x_inc)
x_inc = Dense(256, activation='relu')(x_inc)
x_inc = Dropout(0.4)(x_inc)
outputs_inc = Dense(2, activation='softmax')(x_inc)

model_inception = Model(inputs_inc, outputs_inc)
model_inception.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                        loss='sparse_categorical_crossentropy',
                        metrics=['accuracy'])

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

print("Addestramento InceptionV3 (con EarlyStopping)...")
history_inception = model_inception.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=[early_stopping]
)

inception_model_path = f'{MODELS_DIR}/inception_model'
model_inception.export(inception_model_path)
print(f"Modello InceptionV3 salvato: {inception_model_path}")

### MobileNet V3 small + quantizzazione

In [ ]:
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as v3_preprocess

print("Costruzione MobileNetV3-Small (Ultra-Lite)...")
base_model_v3 = MobileNetV3Small(input_shape=(224, 224, 3), include_top=False, weights='imagenet')

inputs_v3 = tf.keras.Input(shape=(224, 224, 3))
x_v3 = data_augmentation(inputs_v3)
x_v3 = v3_preprocess(x_v3)
x_v3 = base_model_v3(x_v3, training=False)
x_v3 = GlobalAveragePooling2D()(x_v3)
x_v3 = Dense(64, activation='relu')(x_v3)
outputs_v3 = Dense(2, activation='softmax')(x_v3)

model_v3 = Model(inputs_v3, outputs_v3)
model_v3.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Addestramento Flash MobileNetV3 (5 epoche)...")
history_v3 = model_v3.fit(train_dataset, validation_data=val_dataset, epochs=5)

converter_v3 = tf.lite.TFLiteConverter.from_keras_model(model_v3)
converter_v3.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_v3_model = converter_v3.convert()

v3_tflite_path = f'{MODELS_DIR}/mobilenet_v3_small_quant.tflite'
with open(v3_tflite_path, 'wb') as f:
    f.write(tflite_v3_model)

print(f"Modello Ultra-Lite salvato: {v3_tflite_path}")

### Report

In [ ]:
def get_file_size_mb(file_path):
    return os.path.getsize(file_path) / (1024 * 1024) if os.path.exists(file_path) else 0

def get_folder_size_mb(folder_path):
    total_size = 0
    if os.path.exists(folder_path):
        for dirpath, _, filenames in os.walk(folder_path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                if not os.path.islink(fp):
                    total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)

def measure_latency_keras(model, img_tensor, preproc_func=None):
    img = preproc_func(img_tensor.copy()) if preproc_func else img_tensor
    _ = model.predict(img, verbose=0)
    start_time = time.time()
    for _ in range(50):
        _ = model.predict(img, verbose=0)
    end_time = time.time()
    return ((end_time - start_time) / 50) * 1000

def measure_latency_tflite(tflite_path, img_tensor, preproc_func=None):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()

    img = preproc_func(img_tensor.copy()) if preproc_func else img_tensor
    if input_details[0]['dtype'] == np.float32:
        img = img.astype(np.float32)

    interpreter.set_tensor(input_details[0]['index'], img)
    interpreter.invoke()

    start_time = time.time()
    for _ in range(50):
        interpreter.set_tensor(input_details[0]['index'], img)
        interpreter.invoke()
    end_time = time.time()
    return ((end_time - start_time) / 50) * 1000

def evaluate_tflite_model(tflite_path, dataset, preproc_func=None):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    correct_predictions = 0
    total_predictions = 0

    for images, labels in dataset:
        for i in range(images.shape[0]):
            img = images[i:i+1].numpy().copy()
            if preproc_func:
                img = preproc_func(img)

            if input_details[0]['dtype'] == np.float32:
                img = img.astype(np.float32)

            interpreter.set_tensor(input_details[0]['index'], img)
            interpreter.invoke()
            output = interpreter.get_tensor(output_details[0]['index'])

            predicted_label = np.argmax(output)
            if predicted_label == labels[i].numpy():
                correct_predictions += 1
            total_predictions += 1

    return correct_predictions / total_predictions if total_predictions > 0 else 0

size_inc_keras = get_folder_size_mb(inception_model_path)
size_v2_keras = get_folder_size_mb(v2_keras_path)
size_v2_tflite = get_file_size_mb(tflite_path)
size_v3_tflite = get_file_size_mb(v3_tflite_path)

print("Misurazione latenze e valutazione modelli TFLite in corso...")

latency_inc_keras = measure_latency_keras(model_inception, sample_img, inception_preprocess)
latency_v2_keras = measure_latency_keras(model_v2, sample_img, tf.keras.applications.mobilenet_v2.preprocess_input)
latency_v2_tflite = measure_latency_tflite(tflite_path, sample_img, tf.keras.applications.mobilenet_v2.preprocess_input)
latency_v3_tflite = measure_latency_tflite(v3_tflite_path, sample_img, v3_preprocess)

val_acc_inc = max(history_inception.history['val_accuracy'])
val_acc_v2 = max(history_v2.history['val_accuracy'])
val_acc_v3_keras = max(history_v3.history['val_accuracy'])

val_acc_v2_tflite = evaluate_tflite_model(tflite_path, val_dataset, tf.keras.applications.mobilenet_v2.preprocess_input)
val_acc_v3_tflite = evaluate_tflite_model(v3_tflite_path, val_dataset, v3_preprocess)

def plot_comparison(h_edge, h_inception, h_v3):
    epochs_edge = range(len(h_edge.history['val_accuracy']))
    epochs_inc = range(len(h_inception.history['val_accuracy']))
    epochs_v3 = range(len(h_v3.history['val_accuracy']))

    plt.figure(figsize=(20, 8))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_edge, h_edge.history['val_accuracy'], label='Edge (MobileNetV2 Keras)', marker='o')
    plt.plot(epochs_inc, h_inception.history['val_accuracy'], label='Cloud (InceptionV3 Keras)', marker='s')
    plt.plot(epochs_v3, h_v3.history['val_accuracy'], label='Ultra-Lite (MobileNetV3 Keras)', marker='d')
    plt.axhline(y=val_acc_v2_tflite, color='blue', linestyle='--', alpha=0.5, label='V2 TFLite (Int8)')
    plt.axhline(y=val_acc_v3_tflite, color='green', linestyle='--', alpha=0.5, label='V3 TFLite (Int8)')

    plt.title('Accuratezza di Validazione')
    plt.xlabel('Epoche')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(epochs_edge, h_edge.history['val_loss'], label='Edge (MobileNetV2 Keras)', marker='o')
    plt.plot(epochs_inc, h_inception.history['val_loss'], label='Cloud (InceptionV3 Keras)', marker='s')
    plt.plot(epochs_v3, h_v3.history['val_loss'], label='Ultra-Lite (MobileNetV3 Keras)', marker='d')
    plt.title('Perdita (Loss) di Validazione')
    plt.xlabel('Epoche')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plot_path = f'{PLOTS_DIR}/confronto_modelli.png'
    plt.savefig(plot_path)
    print(f"\nGrafico di confronto salvato in: {plot_path}")
    plt.show()

print("\n" + "="*135)
print("REPORT DI COMPARAZIONE COMPLETO: I 4 MODELLI")
print("="*135)
print(f"{'Metrica':<20} | {'Cloud (IncV3 Keras)':<22} | {'Edge (V2 Keras)':<20} | {'Edge Quant (V2 TFLite)':<25} | {'Ultra-Lite (V3 TFLite)'}")
print("-" * 135)

print(f"{'Val Acc Reale':<20} | {val_acc_inc:.4f}                 | {val_acc_v2:.4f}               | {val_acc_v2_tflite:.4f}                   | {val_acc_v3_tflite:.4f}")
print(f"{'Latenza (ms/img)':<20} | {latency_inc_keras:.2f} ms                  | {latency_v2_keras:.2f} ms                | {latency_v2_tflite:.2f} ms                     | {latency_v3_tflite:.2f} ms")
print(f"{'Dimensione su disco':<20} | {size_inc_keras:.2f} MB                 | {size_v2_keras:.2f} MB                | {size_v2_tflite:.2f} MB                      | {size_v3_tflite:.2f} MB")
print("="*135)

plot_comparison(history_v2, history_inception, history_v3)